#### Imports

In [1]:
import stim
import pymatching
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch_geometric.data import Data, Batch
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import add_self_loops, degree
from datetime import datetime
from pathlib import Path
import os

In [2]:
# If CUDA is available, show more details
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"Current device: {torch.cuda.current_device()}")
else:
    print("PyTorch is using CPU only")

CUDA version: 12.1
Number of GPUs: 1
GPU name: NVIDIA GeForce GTX 1080
Current device: 0


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


#### Definitions and Ground Truth

In [4]:
def surface_code_circuit(p, d): # physical error rate, distance
  return stim.Circuit.generated(
    "surface_code:rotated_memory_z",
    rounds=d,
    distance=d,
    after_clifford_depolarization=p,
    after_reset_flip_probability=p,
    before_measure_flip_probability=p,
    before_round_data_depolarization=p)

def count_logical_errors(circuit: stim.Circuit, num_shots: int) -> int:
  # Sample the circuit.
  sampler = circuit.compile_detector_sampler()
  detection_events, observable_flips = sampler.sample(num_shots, separate_observables=True)

  # Configure a decoder using the circuit.
  detector_error_model = circuit.detector_error_model(decompose_errors=True)
  matcher = pymatching.Matching.from_detector_error_model(detector_error_model)

  # Run the decoder.
  predictions = matcher.decode_batch(detection_events)

  # basically compare predictions with observable_flips (what we should have measured)

  # Count the mistakes.
  num_errors = 0
  for shot in range(num_shots):
    actual_for_shot = observable_flips[shot]
    predicted_for_shot = predictions[shot]
    if not np.array_equal(actual_for_shot, predicted_for_shot):
        num_errors += 1
  return num_errors

def ler_mwpm(p, d): # logical error rate, minimum weight perfect matching
  num_shots = 100000
  circuit = surface_code_circuit(p, d)
  num_errors = count_logical_errors(circuit, num_shots)

  return num_errors / num_shots

def plot_mwpm():
  num_shots = 100000
  for d in [3, 5, 7]:
    xs = []
    ys = []
    yerrs = []
    for noise in np.linspace(0.001, 0.008, 8):
      ler = ler_mwpm(noise, d)
      xs.append(noise)
      ys.append(ler)
      yerrs.append(np.sqrt(ler * (1 - ler) / num_shots))
  plt.loglog()
  plt.xlabel("physical error rate")
  plt.ylabel("logical error rate per shot")
  plt.legend()
  plt.show()


#### Dataset Creation


In [5]:
class SurfaceCodeSampler:
    """
    Sampler class for surface code quantum error correction.

    This class generates surface code circuits and creates training datasets
    with configurable error rates.

    Attributes:
        d (int): Code distance
        default_p (float): Default physical error rate
        circuit (stim.Circuit): The generated surface code circuit at default_p
        device (torch.device): Device to use for tensors
    """

    def __init__(self, d: int, p: float = 0.01, device: torch.device = None):
        """
        Initialize the sampler with a surface code circuit.

        Args:
            d (int): Code distance (determines code size and rounds)
            p (float): Default physical error rate (used if not overridden)
            device (torch.device): Device for tensors (defaults to CUDA if available)
        """
        self.d = d
        self.default_p = p
        self.device = device if device else torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # Generate and save the default circuit
        self.circuit = self._generate_circuit(p)

        # Cache circuits for different p values to avoid regenerating
        self._circuit_cache = {p: self.circuit}

        print(f"SurfaceCodeSampler initialized:")
        print(f"  Distance: {d}")
        print(f"  Default error rate: {p}")
        print(f"  Device: {self.device}")
        print(f"  Num detectors: {self.circuit.num_detectors}")

    def _generate_circuit(self, p: float) -> stim.Circuit:
        """Generate a surface code circuit with given error rate."""
        return stim.Circuit.generated(
            "surface_code:rotated_memory_z",
            rounds=self.d,
            distance=self.d,
            after_clifford_depolarization=p,
            after_reset_flip_probability=p,
            before_measure_flip_probability=p,
            before_round_data_depolarization=p
        )

    def _get_circuit(self, p: float) -> stim.Circuit:
        """Get circuit for a given p, using cache if available."""
        if p not in self._circuit_cache:
            self._circuit_cache[p] = self._generate_circuit(p)
        return self._circuit_cache[p]

    def get_circuit(self) -> stim.Circuit:
        """Return the default circuit."""
        return self.circuit

    def sample(self,
               num_samples: int,
               p_values: list[float] = None,
               p_weights: list[float] = None,
               return_p_labels: bool = False) -> tuple:
        """
        Generate training data samples with configurable error rate distribution.

        This function generates syndrome detection data and observable flip labels.
        You can specify multiple error rates with weights to control what percentage
        of the dataset uses each error rate.

        Args:
            num_samples (int): Total number of samples to generate
            p_values (list[float], optional): Array of physical error rates to use.
                                              Defaults to [self.default_p].
            p_weights (list[float], optional): Array of weights (same length as p_values).
                                               Must sum to 1.0. Determines what fraction
                                               of samples use each error rate.
                                               Defaults to [1.0] (all samples at one rate).
            return_p_labels (bool): If True, also return which p was used for each sample.

        Returns:
            tuple: (detections, labels) or (detections, labels, p_indices) if return_p_labels
                - detections: torch.Tensor [num_samples, num_detectors] syndrome measurements (-1 or +1)
                - labels: torch.Tensor [num_samples] observable flip labels (0 or 1)
                - p_indices: torch.Tensor [num_samples] index into p_values for each sample

        Examples:
            # Single error rate (uses default p)
            detections, labels = sampler.sample(10000)

            # Single custom error rate
            detections, labels = sampler.sample(10000, p_values=[0.005], p_weights=[1.0])

            # Mixed error rates: 50% at p=0.001, 30% at p=0.003, 20% at p=0.005
            detections, labels = sampler.sample(
                10000,
                p_values=[0.001, 0.003, 0.005],
                p_weights=[0.5, 0.3, 0.2]
            )
        """
        # Handle defaults
        if p_values is None:
            p_values = [self.default_p]
        if p_weights is None:
            p_weights = [1.0]

        # Validate inputs
        if len(p_values) != len(p_weights):
            raise ValueError(f"p_values and p_weights must have same length. "
                           f"Got {len(p_values)} and {len(p_weights)}")

        weight_sum = sum(p_weights)
        if not np.isclose(weight_sum, 1.0, atol=1e-6):
            raise ValueError(f"p_weights must sum to 1.0, got {weight_sum}")

        # Calculate samples per error rate
        samples_per_p = []
        remaining = num_samples
        for i, weight in enumerate(p_weights):
            if i == len(p_weights) - 1:
                # Last one gets remaining to handle rounding
                n = remaining
            else:
                n = int(round(num_samples * weight))
                remaining -= n
            samples_per_p.append(n)

        # Generate samples for each error rate
        all_detections = []
        all_labels = []
        all_p_indices = []

        for i, (p, n) in enumerate(zip(p_values, samples_per_p)):
            if n <= 0:
                continue

            circuit = self._get_circuit(p)
            sampler = circuit.compile_detector_sampler()
            detections, flips = sampler.sample(n, separate_observables=True)

            # Convert to tensors
            # Convert detections: 0 -> -1, 1 -> +1 for easier reading
            det_np = detections.astype(np.float32) * 2 - 1
            det_tensor = torch.from_numpy(det_np)
            label_tensor = torch.from_numpy(flips.astype(np.float32).flatten())

            all_detections.append(det_tensor)
            all_labels.append(label_tensor)
            all_p_indices.append(torch.full((n,), i, dtype=torch.long))

        # Concatenate all samples
        detections = torch.cat(all_detections, dim=0).to(self.device)
        labels = torch.cat(all_labels, dim=0).to(self.device)
        p_indices = torch.cat(all_p_indices, dim=0).to(self.device)

        # Shuffle the dataset so error rates are mixed
        perm = torch.randperm(num_samples, device=self.device)
        detections = detections[perm]
        labels = labels[perm]
        p_indices = p_indices[perm]

        if return_p_labels:
            return detections, labels, p_indices
        return detections, labels

#### Graph Construction (with Z-Score Normalization)

In [6]:
class SparseGraphNormalized:
    """
    Converts syndrome detections into sparse PyTorch Geometric graphs.

    Same as SparseGraph from gnn.ipynb but with Z-score normalization on edge weights
    to make them suitable for attention mechanisms.

    Node features: [X?, Z?, d_North, d_West, d_time] (5 features)
    Edge weights: Z-score normalized inverse-square distances
    """

    def __init__(self, k_neighbors: int = 6, device: torch.device = None):
        self.k_neighbors = k_neighbors
        self.device = device if device else torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self._coord_cache = {}
        self._feature_cache = {}

        print(f"SparseGraphNormalized initialized:")
        print(f"  K neighbors: {k_neighbors}")
        print(f"  Device: {self.device}")
        print(f"  Edge weight normalization: Z-score (per graph)")

    @staticmethod
    def _infer_distance(num_detectors: int) -> int:
        known = {24: 3, 120: 5, 336: 7, 680: 9, 1168: 11}
        if num_detectors in known:
            return known[num_detectors]
        for d in range(3, 50, 2):
            if (d * d - 1) * d == num_detectors:
                return d
        raise ValueError(f"Cannot infer distance from {num_detectors} detectors")

    @staticmethod
    def _generate_detector_coords(distance: int) -> dict:
        circuit = stim.Circuit.generated(
            "surface_code:rotated_memory_z",
            rounds=distance,
            distance=distance,
            after_clifford_depolarization=0.001
        )
        stim_coords = circuit.get_detector_coordinates()
        detector_coords = {}
        for det_id, coords in stim_coords.items():
            x, y = coords[0], coords[1]
            t = coords[2] if len(coords) > 2 else 0.0
            basis = int(((x + y) / 2) % 2)
            detector_coords[det_id] = [float(x), float(y), float(t), float(basis)]
        return detector_coords

    def _get_coords_and_features(self, num_detectors: int):
        if num_detectors in self._feature_cache:
            return self._feature_cache[num_detectors]

        distance = self._infer_distance(num_detectors)
        if num_detectors not in self._coord_cache:
            self._coord_cache[num_detectors] = self._generate_detector_coords(distance)

        detector_coords = self._coord_cache[num_detectors]
        coords = list(detector_coords.values())
        x_vals, y_vals, t_vals = [c[0] for c in coords], [c[1] for c in coords], [c[2] for c in coords]
        x_min, x_max = min(x_vals), max(x_vals)
        y_min, y_max = min(y_vals), max(y_vals)
        t_min, t_max = min(t_vals), max(t_vals)

        features = []
        for det_id in range(num_detectors):
            coord = detector_coords.get(det_id, [0, 0, 0, 0])
            x, y, t = coord[0], coord[1], coord[2]
            b = coord[3] if len(coord) >= 4 else int(((x + y) / 2) % 2)
            is_x, is_z = (1.0, 0.0) if b == 1 else (0.0, 1.0)
            d_west = (x - x_min) / max(1, x_max - x_min)
            d_north = (y - y_min) / max(1, y_max - y_min)
            d_time = (t - t_min) / max(1, t_max - t_min)
            features.append([is_x, is_z, d_north, d_west, d_time])

        all_features = torch.tensor(features, dtype=torch.float32)
        self._feature_cache[num_detectors] = (detector_coords, all_features)
        return detector_coords, all_features

    def _supremum_distance(self, feat_i: torch.Tensor, feat_j: torch.Tensor) -> float:
        return max(
            abs(feat_i[2] - feat_j[2]).item(),
            abs(feat_i[3] - feat_j[3]).item(),
            abs(feat_i[4] - feat_j[4]).item()
        )

    def _compute_edge_weight(self, sup_dist: float) -> float:
        if sup_dist < 1e-9:
            return 1.0
        return sup_dist ** (-2)

    def to_pyg(self, detections: torch.Tensor, label: torch.Tensor) -> Data:
        """Convert detection sample to PyG Data with z-score normalized edge weights."""
        detections_cpu = detections.cpu()
        num_detectors = detections_cpu.shape[0]
        _, all_features = self._get_coords_and_features(num_detectors)

        fired_mask = detections_cpu > 0
        fired_indices = torch.where(fired_mask)[0].tolist()

        # Handle edge cases
        if len(fired_indices) == 0:
            return Data(
                x=torch.zeros((0, 5), dtype=torch.float32),
                edge_index=torch.zeros((2, 0), dtype=torch.long),
                edge_attr=torch.zeros((0, 1), dtype=torch.float32),
                y=label.cpu().clone()
            )
        if len(fired_indices) == 1:
            return Data(
                x=all_features[fired_indices],
                edge_index=torch.zeros((2, 0), dtype=torch.long),
                edge_attr=torch.zeros((0, 1), dtype=torch.float32),
                y=label.cpu().clone()
            )

        node_features = all_features[fired_indices]
        num_nodes = len(fired_indices)

        # Build edges with k-nearest neighbors
        edges = []
        edge_weights = []
        for i in range(num_nodes):
            distances = [(j, self._supremum_distance(node_features[i], node_features[j]))
                        for j in range(num_nodes) if i != j]
            distances.sort(key=lambda x: x[1])
            for j, sup_dist in distances[:min(self.k_neighbors, len(distances))]:
                edges.append([i, j])
                edge_weights.append(self._compute_edge_weight(sup_dist))

        if len(edges) > 0:
            edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
            edge_attr = torch.tensor(edge_weights, dtype=torch.float32).unsqueeze(1)

            # Z-score normalization of edge weights (per graph)
            edge_attr = (edge_attr - edge_attr.mean()) / (edge_attr.std() + 1e-8)
        else:
            edge_index = torch.zeros((2, 0), dtype=torch.long)
            edge_attr = torch.zeros((0, 1), dtype=torch.float32)

        return Data(x=node_features, edge_index=edge_index, edge_attr=edge_attr, y=label.cpu().clone())

    def batch_to_pyg(self, detections: list, labels: list) -> list:
        """Convert batch of detections to list of PyG Data objects."""
        return [self.to_pyg(det, lbl) for det, lbl in zip(detections, labels)]

#### GAT Model

Using `torch_geometric.nn.models.GAT` with:
- `v2=True` for GATv2Conv (improved attention)
- `edge_dim=1` to incorporate edge weights
- Z-score normalized edge weights from `SparseGraphNormalized`

In [ ]:
from torch_geometric.nn.models import GAT as PyG_GAT
from torch_geometric.nn import global_mean_pool


class GATModel(torch.nn.Module):
    """
    Graph Attention Network for QEC decoding.

    Uses PyG's built-in GAT model as backbone, then adds:
    - Global mean pooling to aggregate node embeddings to graph-level
    - Classification head for binary prediction

    Architecture:
        Input: [N, 5] node features + [E, 1] edge features
        → GAT backbone (GATv2Conv layers with attention)
        → Global mean pool → [batch_size, hidden_dim]
        → FC layers → sigmoid → probability
    """

    def __init__(
        self,
        in_channels: int = 5,
        hidden_channels: int = 32,  # Per-head dimension
        num_layers: int = 4,
        heads: int = 4,
        dropout: float = 0.1,
        edge_dim: int = 1,
    ):
        super().__init__()
        self.in_channels = in_channels
        self.hidden_channels = hidden_channels
        self.num_layers = num_layers
        self.heads = heads

        # GAT backbone using PyG's built-in model
        # v2=True uses GATv2Conv (improved attention mechanism)
        # concat=True means output dim = hidden_channels * heads
        self.gat = PyG_GAT(
            in_channels=in_channels,
            hidden_channels=hidden_channels,
            num_layers=num_layers,
            v2=True,
            heads=heads,
            dropout=dropout,
            act='silu',
            norm='batch',
            edge_dim=edge_dim,
        )

        # Output dimension after GAT: hidden_channels * heads (due to concat)
        # For final layer, PyG's GAT averages heads, so output = hidden_channels
        # But intermediate layers concat, so we need to check actual output
        # PyG GAT: last layer uses heads=1 by default when out_channels not set
        gat_out_dim = hidden_channels

        # Classification head
        self.fc1 = torch.nn.Linear(gat_out_dim, 64)
        self.fc2 = torch.nn.Linear(64, 1)

    def forward(self, data) -> torch.Tensor:
        x, edge_index = data.x, data.edge_index
        edge_attr = data.edge_attr if hasattr(data, 'edge_attr') and data.edge_attr is not None else None
        batch = data.batch if hasattr(data, 'batch') and data.batch is not None else torch.zeros(x.size(0), dtype=torch.long, device=x.device)

        # Get actual batch size from data
        batch_size = int(data.num_graphs) if hasattr(data, 'num_graphs') else (int(batch.max().item()) + 1 if x.size(0) > 0 else 1)

        # Handle completely empty batch (no nodes at all)
        if x.size(0) == 0:
            return torch.full((batch_size, 1), 0.5, device=data.y.device if hasattr(data, 'y') else 'cpu')

        # GAT forward pass: returns node embeddings
        x = self.gat(x, edge_index, edge_attr=edge_attr)

        # Global mean pooling: aggregate to graph-level
        # This handles empty graphs within the batch by producing zeros for them
        x_pooled = global_mean_pool(x, batch, size=batch_size)

        # Classification head
        x = self.fc1(x_pooled)
        x = F.silu(x)
        x = self.fc2(x)
        x = torch.sigmoid(x)

        return x


class GATDecoder:
    """
    GAT wrapper class with model lifecycle management.

    Provides init, train, save, load, and evaluation functionality
    with logging and experiment tracking.

    Follows the same pattern as GCN wrapper from gnn.ipynb.
    """

    MODELS_DIR = Path("models/gat")

    def __init__(
        self,
        nickname: str = "gat_decoder",
        in_channels: int = 5,
        hidden_channels: int = 32,
        num_layers: int = 4,
        heads: int = 4,
        dropout: float = 0.1,
        device: torch.device = None,
    ):
        self.nickname = nickname
        self.device = device if device else torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self._loaded_from = None

        self._config = {
            'in_channels': in_channels,
            'hidden_channels': hidden_channels,
            'num_layers': num_layers,
            'heads': heads,
            'dropout': dropout,
        }

        self.model = GATModel(
            in_channels=in_channels,
            hidden_channels=hidden_channels,
            num_layers=num_layers,
            heads=heads,
            dropout=dropout,
        ).to(self.device)

        self.MODELS_DIR.mkdir(parents=True, exist_ok=True)
        self.training_history = []

        print(f"GATDecoder initialized: {self}")

    def train(
        self,
        graphs: list,
        epochs: int = 50,
        batch_size: int = 64,
        lr: float = 1e-3,
        val_split: float = 0.1,
        verbose: bool = True,
        log_every: int = 5,
    ) -> dict:
        """
        Train the model with validation and logging.

        Args:
            graphs: List of PyG Data objects
            epochs: Number of training epochs
            batch_size: Batch size
            lr: Learning rate
            val_split: Fraction of data for validation
            verbose: Print progress
            log_every: Print every N epochs

        Returns:
            Dictionary with training history
        """
        from torch_geometric.loader import DataLoader
        import random

        # Split into train/val
        shuffled = graphs.copy()
        random.shuffle(shuffled)
        val_size = int(len(shuffled) * val_split)
        val_graphs = shuffled[:val_size]
        train_graphs = shuffled[val_size:]

        train_loader = DataLoader(train_graphs, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_graphs, batch_size=batch_size, shuffle=False) if val_size > 0 else None

        optimizer = torch.optim.Adam(self.model.parameters(), lr=lr)
        loss_fn = torch.nn.BCELoss()

        history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

        if verbose:
            print(f"Training on {len(train_graphs)} samples, validating on {len(val_graphs)}")
            print("-" * 50)

        for epoch in range(epochs):
            # Training
            self.model.train()
            total_loss = 0.0
            num_batches = 0

            for batch_data in train_loader:
                batch_data = batch_data.to(self.device)
                pred = self.model(batch_data)
                y = batch_data.y.float().view(-1, 1)
                loss = loss_fn(pred, y)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                total_loss += loss.item()
                num_batches += 1

            avg_train_loss = total_loss / max(num_batches, 1)
            history['train_loss'].append(avg_train_loss)

            # Validation
            val_loss, val_acc = 0.0, 0.0
            if val_loader:
                self.model.eval()
                val_total_loss = 0.0
                val_correct = 0
                val_total = 0

                with torch.no_grad():
                    for batch_data in val_loader:
                        batch_data = batch_data.to(self.device)
                        pred = self.model(batch_data)
                        y = batch_data.y.float().view(-1, 1)
                        val_total_loss += loss_fn(pred, y).item()

                        binary_pred = (pred >= 0.5).float()
                        val_correct += (binary_pred == y).sum().item()
                        val_total += y.size(0)

                val_loss = val_total_loss / len(val_loader)
                val_acc = val_correct / val_total if val_total > 0 else 0.0

            history['val_loss'].append(val_loss)
            history['val_acc'].append(val_acc)

            # Logging
            if verbose and (epoch + 1) % log_every == 0:
                print(f"Epoch {epoch+1:3d}/{epochs} | "
                      f"Train Loss: {avg_train_loss:.4f} | "
                      f"Val Loss: {val_loss:.4f} | "
                      f"Val Acc: {val_acc:.4f}")

        self.training_history = history

        if verbose:
            print("-" * 50)
            print(f"Training complete. Final val accuracy: {history['val_acc'][-1]:.4f}")

        return history

    def save(self, name: str) -> Path:
        """Save model with timestamp."""
        timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
        filename = f"{name}_{timestamp}.pt"
        filepath = self.MODELS_DIR / filename
        self.MODELS_DIR.mkdir(parents=True, exist_ok=True)

        save_dict = {
            'state_dict': self.model.state_dict(),
            'config': self._config,
            'nickname': self.nickname,
            'timestamp': timestamp,
            'training_history': self.training_history,
        }
        torch.save(save_dict, filepath)
        print(f"Model saved to: {filepath}")
        return filepath

    def load(self, filepath: str) -> 'GATDecoder':
        """Load model from disk."""
        path = Path(filepath)
        if not path.exists():
            path = self.MODELS_DIR / filepath
        if not path.exists():
            raise FileNotFoundError(f"Model file not found: {filepath}")

        save_dict = torch.load(path, map_location=self.device, weights_only=False)
        config = save_dict['config']
        self._config = config

        self.model = GATModel(
            in_channels=config['in_channels'],
            hidden_channels=config['hidden_channels'],
            num_layers=config['num_layers'],
            heads=config['heads'],
            dropout=config.get('dropout', 0.1),
        ).to(self.device)

        self.model.load_state_dict(save_dict['state_dict'])
        self.model.eval()
        self._loaded_from = path.name
        self.nickname = save_dict.get('nickname', self.nickname)
        self.training_history = save_dict.get('training_history', [])

        print(f"Model loaded: {self}")
        return self

    def predict(self, data) -> torch.Tensor:
        """Run inference."""
        self.model.eval()
        with torch.no_grad():
            data = data.to(self.device)
            return self.model(data)

    def __repr__(self) -> str:
        loaded_info = f", loaded_from='{self._loaded_from}'" if self._loaded_from else ""
        return (f"GATDecoder(nickname='{self.nickname}', "
                f"hidden={self._config['hidden_channels']}, "
                f"layers={self._config['num_layers']}, "
                f"heads={self._config['heads']}"
                f"{loaded_info})")

#### Training

In [8]:
# Step 1: Initialize sampler
sampler = SurfaceCodeSampler(d=3, p=0.005, device=device)

# Step 2: Generate training data with mixed error rates
p_values = [0.001, 0.002, 0.003, 0.004, 0.005]
p_weights = [0.2, 0.2, 0.2, 0.2, 0.2]

num_samples = 10000
train_detections, train_labels, p_indices = sampler.sample(
    num_samples=num_samples,
    p_values=p_values,
    p_weights=p_weights,
    return_p_labels=True
)

print(f"\nTraining data generated:")
print(f"  Detections shape: {train_detections.shape}")
print(f"  Labels distribution: {train_labels.sum().item():.0f} positive / {num_samples} total")

SurfaceCodeSampler initialized:
  Distance: 3
  Default error rate: 0.005
  Device: cuda
  Num detectors: 24

Training data generated:
  Detections shape: torch.Size([10000, 24])
  Labels distribution: 648 positive / 10000 total


In [9]:
# Step 3: Convert to graphs with z-score normalized edge weights
graph_builder = SparseGraphNormalized(k_neighbors=6, device=device)
graphs = graph_builder.batch_to_pyg(train_detections, train_labels)

print(f"Generated {len(graphs)} graphs")

# Inspect a sample graph
for i, g in enumerate(graphs):
    if g.x.shape[0] > 2:  # Find a graph with some nodes
        print(f"\nSample graph {i}:")
        print(f"  Nodes: {g.x.shape[0]}, Edges: {g.edge_index.shape[1]}")
        print(f"  Edge weights (z-scored): min={g.edge_attr.min():.3f}, max={g.edge_attr.max():.3f}, mean={g.edge_attr.mean():.3f}")
        break

SparseGraphNormalized initialized:
  K neighbors: 6
  Device: cuda
  Edge weight normalization: Z-score (per graph)
Generated 10000 graphs

Sample graph 2:
  Nodes: 4, Edges: 12
  Edge weights (z-scored): min=-1.354, max=0.677, mean=0.000


In [10]:
# Step 4: Initialize and train GAT model
gat = GATDecoder(
    nickname="qec_gat_v1",
    hidden_channels=32,  # Per-head dimension (32 * 4 heads = 128 total)
    num_layers=4,
    heads=4,
    dropout=0.1,
)

# Train with validation
history = gat.train(
    graphs,
    epochs=50,
    batch_size=64,
    lr=1e-3,
    val_split=0.1,
    log_every=5,
)

GATDecoder initialized: GATDecoder(nickname='qec_gat_v1', hidden=32, layers=4, heads=4)
Training on 9000 samples, validating on 1000
--------------------------------------------------


ValueError: Using a target size (torch.Size([64, 1])) that is different to the input size (torch.Size([60, 1])) is deprecated. Please ensure they have the same size.

In [ ]:
# Step 5: Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss curves
axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('BCE Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curve
axes[1].plot(history['val_acc'], label='Val Accuracy', color='green')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Step 6: Save the trained model
gat.save("experiment_1")

#### Evaluation & Comparison with MWPM

In [ ]:
def test_gat_accuracy(
    gat: GATDecoder,
    sampler: SurfaceCodeSampler,
    graph_builder: SparseGraphNormalized,
    num_samples: int = 5000,
    p_values: list[float] = None,
    p_weights: list[float] = None,
    threshold: float = 0.5,
    compare_mwpm: bool = True,
    verbose: bool = True
) -> dict:
    """
    Test GAT accuracy and compare with MWPM decoder.
    """
    from torch_geometric.data import Batch

    gat.model.eval()

    if p_values is None:
        p_values = [sampler.default_p]
    if p_weights is None:
        p_weights = [1.0 / len(p_values)] * len(p_values)

    # Generate test data
    test_detections, test_labels, p_indices = sampler.sample(
        num_samples=num_samples,
        p_values=p_values,
        p_weights=p_weights,
        return_p_labels=True
    )

    # Convert to graphs
    test_graphs = graph_builder.batch_to_pyg(test_detections, test_labels)

    # Run predictions in batches
    all_preds = []
    batch_size = 64

    with torch.no_grad():
        for i in range(0, len(test_graphs), batch_size):
            batch = Batch.from_data_list(test_graphs[i:i+batch_size])
            batch = batch.to(gat.device)
            pred = gat.model(batch)
            all_preds.append(pred.cpu())

    predictions = torch.cat(all_preds, dim=0).squeeze()
    binary_preds = (predictions >= threshold).float()
    labels_cpu = test_labels.cpu()

    # Metrics
    correct = (binary_preds == labels_cpu).sum().item()
    accuracy = correct / num_samples

    tp = ((binary_preds == 1) & (labels_cpu == 1)).sum().item()
    fp = ((binary_preds == 1) & (labels_cpu == 0)).sum().item()
    fn = ((binary_preds == 0) & (labels_cpu == 1)).sum().item()
    tn = ((binary_preds == 0) & (labels_cpu == 0)).sum().item()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    results = {
        'num_samples': num_samples,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'logical_error_rate_gat': 1.0 - accuracy,
    }

    # Per-error-rate breakdown
    per_p_accuracy = {}
    for i, p in enumerate(p_values):
        mask = (p_indices.cpu() == i)
        if mask.sum() > 0:
            correct_p = (binary_preds[mask] == labels_cpu[mask]).sum().item()
            per_p_accuracy[p] = correct_p / mask.sum().item()
    results['per_p_accuracy'] = per_p_accuracy

    # MWPM comparison
    if compare_mwpm:
        mwpm_errors = 0
        for i, p in enumerate(p_values):
            mask = (p_indices.cpu() == i)
            n = mask.sum().item()
            if n > 0:
                circuit = sampler._get_circuit(p)
                mwpm_errors += count_logical_errors(circuit, n)

        mwpm_accuracy = 1.0 - (mwpm_errors / num_samples)
        results['mwpm_accuracy'] = mwpm_accuracy
        results['logical_error_rate_mwpm'] = mwpm_errors / num_samples
        results['improvement_over_mwpm'] = accuracy - mwpm_accuracy

    if verbose:
        print("=" * 60)
        print("GAT Decoder Test Results")
        print("=" * 60)
        print(f"Test samples: {num_samples}")
        print(f"Error rates: {p_values}")
        print("-" * 60)
        print(f"GAT Accuracy: {accuracy:.4f} ({correct}/{num_samples})")
        print(f"GAT Logical Error Rate: {results['logical_error_rate_gat']:.4f}")
        print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")
        print("-" * 60)
        print("Per-error-rate accuracy:")
        for p, acc in per_p_accuracy.items():
            print(f"  p={p}: {acc:.4f}")
        if compare_mwpm:
            print("-" * 60)
            print(f"MWPM Accuracy: {mwpm_accuracy:.4f}")
            print(f"GAT vs MWPM: {results['improvement_over_mwpm']:+.4f}")
        print("=" * 60)

    return results

In [ ]:
# Run evaluation
results = test_gat_accuracy(
    gat=gat,
    sampler=sampler,
    graph_builder=graph_builder,
    num_samples=5000,
    p_values=[0.001, 0.003, 0.005],
    p_weights=[0.33, 0.34, 0.33],
    compare_mwpm=True
)

#### Next Steps & Hyperparameter Tuning

**Things to try:**
1. **Increase training data**: Try 50k-100k samples
2. **Adjust architecture**: More/fewer layers, different head counts
3. **Learning rate schedule**: Add warmup or decay
4. **Different distances**: Train on d=5 or d=7
5. **Regularization**: Increase dropout if overfitting

In [ ]:
# Hyperparameter experiments template
# Uncomment and modify to try different configurations

# EXPERIMENT: More layers
# gat_deep = GATDecoder(nickname="gat_deep", num_layers=6, hidden_channels=32, heads=4)
# gat_deep.train(graphs, epochs=50)

# EXPERIMENT: More heads
# gat_heads = GATDecoder(nickname="gat_8heads", num_layers=4, hidden_channels=16, heads=8)
# gat_heads.train(graphs, epochs=50)

# EXPERIMENT: Larger hidden dimension
# gat_wide = GATDecoder(nickname="gat_wide", num_layers=4, hidden_channels=64, heads=4)
# gat_wide.train(graphs, epochs=50)

# EXPERIMENT: Higher dropout
# gat_reg = GATDecoder(nickname="gat_reg", num_layers=4, hidden_channels=32, heads=4, dropout=0.3)
# gat_reg.train(graphs, epochs=50)